# **Detección de barras en galaxias - Medición de barra detectada**
## **Proyecto integrador MNA**
#### **Integrantes**
 - Jonathan Jesús Marmolejo Hernández - A01795195
 - Isaid Posadas Oropeza - A01795015
 - Luis Daniel Ortega Muñoz - A01795197


#### **Introducción**
A partir de la identificación y clasificación previa de galaxias tipo barrada, se define la ruta en donde se encuentran ubicadas las imagenes de galaxias y el archivo .scv que enlista únicamente las galaxias barradas identificadas previamente. Este programa toma el archivo .scv previamente coemtado, como indice para calcular el tamaño de barras identificadas en la galaxia.

In [ ]:
# Importar librerías necesarias
import pandas as pd
import cv2
import numpy as np
import os

In [ ]:
# Conexión con drive en donde estan ubicadas las imagenes de galaxias
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Definición de ruta en donde este ubicado el archivo .csv que enliste las imagenes identificadas como galaxias barradas
# La única variable obligatoria en el archivo es la variable "name", que describa el nombre de la imagen, ya que servira
# como indice para buscar dicha imagene en la ruta de drive.
csv_path = '/content/dataset2.csv'
df = pd.read_csv(csv_path, encoding='ISO-8859-1')

In [ ]:
df.head()

,name,objra,objdec,Bars,Description
0,manga-10001-12702,133.685670,57.480250,0.50,Barra mezclada con los brazos de la galaxia
1,manga-10001-12703,136.017160,57.092329,0.50,Barra mezclada con los brazos de la galaxia
2,manga-10001-3703,134.591499,57.684965,0.25,Barra dÃÂ©bil
3,manga-10001-6102,132.653992,57.359668,1.00,Barra definida y de gran tamaÃÂ±o relativo a ...
4,manga-10001-6103,134.008123,57.390964,1.00,Barra definida y de gran tamaÃÂ±o relativo a ...


In [ ]:
# Cálculo de variables para definición de elipse.

def calcular_tamano_barra(imagen_path):
    try:
        img = cv2.imread(imagen_path)

        # Proceso para zoom de imagen
        zoom_factor = 3
        threshold = 20
        h, w, _ = img.shape
        cx, cy = w // 2, h // 2

        # Cálculo del centro con base en el zoom deseado
        zoom_w = int(w / zoom_factor)
        zoom_h = int(h / zoom_factor)
        x1 = max(cx - zoom_w // 2, 0)
        x2 = min(cx + zoom_w // 2, w)
        y1 = max(cy - zoom_h // 2, 0)
        y2 = min(cy + zoom_h // 2, h)

        img_zoom = img[y1:y2, x1:x2]

        # Ajustes de nitidez de la imagena para mejorar la definicipon de la elipse
        gray = cv2.cvtColor(img_zoom, cv2.COLOR_RGB2GRAY)
        h, w,_ = img_zoom.shape
        cx, cy = w // 2, h // 2
        _, thresh = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY_INV)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Seleccionar el contorno más cercano al centro
        min_dist = float('inf')
        selected_cnt = None

        for cnt in contours:
            if len(cnt) >= 5:
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cX = int(M["m10"] / M["m00"])
                    cY = int(M["m01"] / M["m00"])
                    dist = np.sqrt((cX - cx)**2 + (cY - cy)**2)
                    if dist < min_dist:
                        min_dist = dist
                        selected_cnt = cnt

        # Crear copia para dibujar las líneas, centro y elipse
        canvas = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

        if selected_cnt is not None:

            # Cálclulo e impresión de elipse
            ellipse = cv2.fitEllipse(selected_cnt)
            (x, y), (MA, ma), angle = ellipse
            cv2.ellipse(canvas, ellipse, (255, 0, 0), 1, cv2.LINE_AA)

            # Cálculo de ejes
            a = MA / 2
            b = ma / 2

            # Cálclulo e impresión de línea
            theta = np.deg2rad(angle)
            pt1 = (int(x - b * np.sin(theta)), int(y + b * np.cos(theta)))
            pt2 = (int(x + b * np.sin(theta)), int(y - b * np.cos(theta)))

            cv2.line(canvas, pt1, pt2, (255,0,0), 1, cv2.LINE_AA)

            # Impresión de centro
            cv2.circle(canvas, (int(x), int(y)), 2, (255,0,0), -1)

            # Cálculo e impresión de longitud de línea
            line_length_px = int(np.linalg.norm(np.array(pt1) - np.array(pt2)))
            text = f"{line_length_px} px"
            mid_point = (pt2[0], pt2[1] -10)
            cv2.putText(canvas,text,mid_point,cv2.FONT_HERSHEY_SIMPLEX,0.3,(255,0,0),1,cv2.LINE_AA)

            ancho_barra = line_length_px

            return round(line_length_px, 2), round(angle, 2), round(MA, 2), round(ma, 2), round(x, 2), round(y, 2)

        else:

            return None, None, None, None, None, None

    except:

        return None, None, None, None, None, None

# Ruta de imágenes insumo
ruta_imagenes = '/content/drive/MyDrive/Proyecto integrador/imagenes/Procesadas'

tamanos_barra, angulos, ejes_mayores, ejes_menores, centros_x, centros_y = [], [], [], [], [], []

for name in df['name']:
    ruta = os.path.join(ruta_imagenes, f"{name}.png")
    if os.path.exists(ruta):

        tamano, angle, MA, ma, x, y = calcular_tamano_barra(ruta)

    else:

        tamano, angle, MA, ma, x, y = None, None, None, None, None, None

    tamanos_barra.append(tamano)
    angulos.append(angle)
    ejes_mayores.append(MA)
    ejes_menores.append(ma)
    centros_x.append(x)
    centros_y.append(y)


# Agregar columna al DataFrame
df['medida_barra'] = tamanos_barra
df['angulo'] = angulos
df['eje_menor'] = ejes_mayores
df['eje_mayor'] = ejes_menores

# Guardar nuevo archivo CSV con columna de tamaño
df.to_csv('/content/drive/MyDrive/Proyecto integrador/dataset2_con_tamano.csv', index=False, encoding='utf-8')

In [ ]:
# Validación de salida

df.head()

,name,objra,objdec,Bars,Description,medida_barra,angulo,eje_menor,eje_mayor
0,manga-10001-12702,133.685670,57.480250,0.50,Barra mezclada con los brazos de la galaxia,86,167.03,42.69,85.83
1,manga-10001-12703,136.017160,57.092329,0.50,Barra mezclada con los brazos de la galaxia,113,130.66,34.88,113.43
2,manga-10001-3703,134.591499,57.684965,0.25,Barra dÃÂ©bil,61,15.14,28.33,60.65
3,manga-10001-6102,132.653992,57.359668,1.00,Barra definida y de gran tamaÃÂ±o relativo a ...,78,110.50,37.14,78.50
4,manga-10001-6103,134.008123,57.390964,1.00,Barra definida y de gran tamaÃÂ±o relativo a ...,104,147.06,69.28,104.73
